In [0]:
from pyspark.sql import functions as F

# Lê a tabela tratada e seleciona os jogos classificados no período
partidas_analise = (
    spark.table("workspace.mvp_gramados.silver_partidas_gramados")
    .filter(F.col("status_gramado") == "dentro_do_periodo")
    .withColumn("ano", F.year("data"))
    .withColumn(
        "total_gols",
        F.col("mandante_Placar") + F.col("visitante_Placar")
    )
    .withColumn(
        "vitoria_mandante",
        F.when(
            F.col("mandante_Placar") > F.col("visitante_Placar"), 1
        ).otherwise(0)
    )
)

print("Partidas incluídas na análise:", partidas_analise.count())

display(
    partidas_analise.select(
        "data", "mandante", "visitante", "tipo_gramado",
        "mandante_Placar", "visitante_Placar",
        "total_gols", "vitoria_mandante"
    ).orderBy("data", "ID").limit(10)
)

In [0]:
# Agrupa as partidas por tipo de gramado
resumo_gramado = (
    partidas_analise
    .groupBy("tipo_gramado")
    .agg(
        F.count("*").alias("quantidade_partidas"),
        F.avg("total_gols").alias("media_gols"),
        F.sum("vitoria_mandante").alias("vitorias_mandante"),
        F.avg("vitoria_mandante").alias("taxa_vitoria_mandante")
    )
)

# Persiste o resultado na camada gold
resumo_gramado.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_gramados.gold_resumo_gramado")

# Lê a tabela salva e formata os números para apresentação
display(spark.sql("""
    SELECT
        tipo_gramado,
        quantidade_partidas,
        ROUND(media_gols, 3) AS media_gols,
        vitorias_mandante,
        ROUND(taxa_vitoria_mandante * 100, 2)
            AS vitorias_mandante_pct
    FROM workspace.mvp_gramados.gold_resumo_gramado
    ORDER BY tipo_gramado
"""))

In [0]:
resumo_ano_gramado = (
    partidas_analise
    .groupBy("ano", "tipo_gramado")
    .agg(
        F.count("*").alias("quantidade_partidas"),
        F.avg("total_gols").alias("media_gols"),
        F.sum("vitoria_mandante").alias("vitorias_mandante"),
        F.avg("vitoria_mandante").alias("taxa_vitoria_mandante")
    )
)

resumo_ano_gramado.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_gramados.gold_resumo_ano_gramado")

display(spark.sql("""
    SELECT
        ano,
        tipo_gramado,
        quantidade_partidas,
        ROUND(media_gols, 3) AS media_gols,
        vitorias_mandante,
        ROUND(taxa_vitoria_mandante * 100, 2)
            AS vitorias_mandante_pct
    FROM workspace.mvp_gramados.gold_resumo_ano_gramado
    ORDER BY ano, tipo_gramado
"""))

In [0]:
resumo_clube_gramado = (
    partidas_analise
    .groupBy("mandante", "tipo_gramado")
    .agg(
        F.count("*").alias("quantidade_partidas"),
        F.avg("total_gols").alias("media_gols"),
        F.sum("vitoria_mandante").alias("vitorias_mandante"),
        F.avg("vitoria_mandante").alias("taxa_vitoria_mandante")
    )
)

resumo_clube_gramado.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_gramados.gold_resumo_clube_gramado")

display(spark.sql("""
    SELECT
        mandante,
        tipo_gramado,
        quantidade_partidas,
        ROUND(media_gols, 3) AS media_gols,
        vitorias_mandante,
        ROUND(taxa_vitoria_mandante * 100, 2)
            AS vitorias_mandante_pct
    FROM workspace.mvp_gramados.gold_resumo_clube_gramado
    ORDER BY tipo_gramado, quantidade_partidas DESC, mandante
"""))